# 🏦 Loan Approval Prediction + SHAP Explainability
This notebook trains a RandomForest on the German Credit dataset and uses SHAP to explain predictions.
It includes fast SHAP and local/global explanations.

In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import shap

# Load data
def load_data():
    try:
        from sklearn.datasets import fetch_openml
        df = fetch_openml("credit-g", version=1, as_frame=True).frame
        df["target"] = (df["class"] == "good").astype(int)
        df = df.drop(columns=["class"])
        src = "openml"
    except Exception:
        rng = np.random.RandomState(42)
        n = 400
        df = pd.DataFrame({
            "duration": rng.randint(4, 60, size=n),
            "amount": rng.randint(500, 20000, size=n),
            "age": rng.randint(18, 75, size=n),
            "employment": rng.choice(["unemployed","<1yr","1-4yrs","4-7yrs",">=7yrs"], size=n, p=[0.1,0.2,0.35,0.2,0.15]),
            "housing": rng.choice(["own","rent","free"], size=n, p=[0.6,0.35,0.05]),
            "savings": rng.choice(["little","moderate","rich"], size=n, p=[0.6,0.3,0.1]),
            "purpose": rng.choice(["car","furniture/equipment","radio/tv","education","business","domestic appliances"], size=n),
        })
        score = (df["amount"]/20000) + (df["duration"]/60) - (df["savings"].map({"little":0.2,"moderate":0.0,"rich":-0.2}))
        y = (score < 0.8).astype(int)
        df["target"] = y
        src = "synthetic"
    return df, src

df, source = load_data()
print("Data source:", source, "| Shape:", df.shape)

target_col = "target"
X = df.drop(columns=[target_col])
y = df[target_col]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Columns by type
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

# Preprocessor
preprocess = ColumnTransformer(transformers=[
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

# Model
pipe = Pipeline([
    ("pre", preprocess),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])
pipe.fit(X_train, y_train)

# Evaluate
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# SHAP (fast mode)
# Feature names after preprocessing
cat_transformer = pipe.named_steps["pre"].named_transformers_["cat"]
oh = cat_transformer.named_steps["oh"]
feature_names = num_cols + list(oh.get_feature_names_out(cat_cols))

# Sample for speed
sample_size = min(400, len(X_test))
sample_idx = np.random.RandomState(42).choice(len(X_test), size=sample_size, replace=False)
X_sample = X_test.iloc[sample_idx]

X_train_trans = pipe.named_steps["pre"].transform(X_train)
X_sample_trans = pipe.named_steps["pre"].transform(X_sample)

explainer = shap.Explainer(pipe.named_steps["clf"], X_train_trans, algorithm="tree")
shap_values = explainer(X_sample_trans, check_additivity=False)

# Global plots
plt.figure(figsize=(6,4))
shap.summary_plot(shap_values, X_sample_trans, feature_names=feature_names, show=False)
plt.show()

plt.figure(figsize=(6,4))
shap.summary_plot(shap_values, X_sample_trans, feature_names=feature_names, plot_type="bar", show=False)
plt.show()

# Local explanation for one row
x_row = X_test.iloc[[0]]
x_row_trans = pipe.named_steps["pre"].transform(x_row)
sv_row = explainer(x_row_trans, check_additivity=False)
vals = sv_row.values[0]
abs_idx = np.argsort(np.abs(vals))[::-1][:10]
import pandas as pd
top_df = pd.DataFrame({"feature": np.array(feature_names)[abs_idx], "shap_value": vals[abs_idx]})
print("\nTop per-row contributors:\n", top_df)
